## 設備時間序列 Log 處理與 Parquet 大數據讀寫優化

- 目標：掌握設備時間序列（Time-Series）日誌的**特徵與重採樣（Resampling）**技巧。精通大數據優化策略，利用 **Apache Parquet 欄位式儲存大幅縮減磁碟空間**，並透過 **Pandas Chunk 機制實現「超低記憶體佔用」的巨量資料分批讀取 Pipeline**。


### 1. 設備 Log 時間序列處理與重採樣 (Resampling)

- 實作：高頻測試機台在運作時，感測器會以秒級甚至毫秒級（毫秒）的高頻率噴出時間序列 Log（如探針床溫度、針壓）。為了與良率或 MES 數據對齊，我們需要將這些密集的時間序列資料按「分鐘」或「小時」進行重採樣與特徵聚合（如計算每分鐘的平均溫度與最大壓力）。


In [ ]:
import pandas as pd
import numpy as np

# 模擬 5 分鐘內、每秒噴出 1 筆的高頻機台感測器時間序列 Log (共 300 筆)
np.random.seed(42)
time_index = pd.date_range(start="2026-08-04 14:00:00", periods=300, freq="s")

raw_log_df = pd.DataFrame(
    {
        "timestamp": time_index,
        "probe_temperature_c": np.random.normal(loc=45.0, scale=0.5, size=300),
        "contact_force_g": np.random.normal(loc=120.0, scale=2.0, size=300),
    }
)

# 將時間戳欄位設定為索引 (DatetimeIndex) — 這是進行時間序列操作的核心步驟
raw_log_df.set_index("timestamp", inplace=True)
print("原始秒級高頻 Log 前 3 行:")
print(raw_log_df.head(3))

# 執行降採樣 (Downsampling) 與聚合：將秒級資料轉化為「每 1 分鐘」的統計指標
# '1T' 代表 1 Minute，我們對溫度求平均(mean)，對針壓求最大值(max)
resampled_df = (
    raw_log_df.resample("1T")
    .agg({"probe_temperature_c": "mean", "contact_force_g": "max"})
    .rename(
        columns={
            "probe_temperature_c": "avg_temp_1m",
            "contact_force_g": "max_force_1m",
        }
    )
)

print("\n 1分鐘重採樣（Resampling）聚合結果:")
print(resampled_df)

### 2. 大量晶圓資料的 Parquet 欄位式儲存優化

- 實作：傳統上產線喜歡把歷史資料存成 CSV 檔，但 CSV 是文字檔，不僅佔用空間，讀取特定欄位時還必須把整張大表載入。Parquet 格式 是**二進位的欄位式儲存（Columnar Storage）**，內建高壓縮比，且支援「**欄位裁剪（Column Projection）**」，非常適合用來儲存巨量晶圓測試資料。


In [ ]:
import os

# 模擬一個較大的晶圓測試歷史資料集 (10 萬行，模擬多片晶圓的 Die 測試點位)
num_rows = 100000
huge_wafer_df = pd.DataFrame(
    {
        "lot_id": np.random.choice(["LOT_X01", "LOT_X02", "LOT_X03"], size=num_rows),
        "wafer_no": np.random.randint(1, 26, size=num_rows),
        "die_x": np.random.randint(0, 100, size=num_rows),
        "die_y": np.random.randint(0, 100, size=num_rows),
        "test_value": np.random.uniform(24.0, 30.0, size=num_rows),
        "bin_code": np.random.choice([1, 1, 1, 1, 2, 3], size=num_rows),  # 1 代表 Pass
    }
)

# --- 分別儲存為 CSV 與 Parquet 格式，對比檔案大小 ---
csv_filename = "large_wafer_data.csv"
parquet_filename = "large_wafer_data.parquet"

huge_wafer_df.to_csv(csv_filename, index=False)
huge_wafer_df.to_parquet(parquet_filename, index=False, compression="snappy")

csv_size = os.path.getsize(csv_filename) / (1024 * 1024)
parquet_size = os.path.getsize(parquet_filename) / (1024 * 1024)

print("=" * 50)
print(f"傳統 CSV 檔案大小 : {csv_size:.2f} MB")
print(f"Snappy Parquet 大小: {parquet_size:.2f} MB")
print(f"檔案空間縮減率     : {(1 - parquet_size / csv_size) * 100:.1f}%")
print("=" * 50)

# --- 欄位裁剪優化：如果機器學習模型只需要 'test_value' 和 'bin_code' 欄位 ---
# Parquet 可以只讀這兩個欄位，而不用把整個檔案載入記憶體
df_subset = pd.read_parquet(parquet_filename, columns=["test_value", "bin_code"])
print(f"欄位裁剪讀取成功，載入的 DataFrame 欄位: {list(df_subset.columns)}")

# 清理實驗檔案
if os.path.exists(csv_filename):
    os.remove(csv_filename)
if os.path.exists(parquet_filename):
    os.remove(parquet_filename)

### 3. Pandas Chunk 機制：防止記憶體溢出 (OOM) 的分批讀取

- 實作：當需要解析一個超大 CSV（例如 10GB 的產線歷史機台日誌）時，直接 pd.read_csv() 會瞬間導致伺服器 OOM（Out of Memory）崩潰。我們必須使用 **chunksize 參數，將其轉化為迭代器**，實現「細水長流」的分批清洗 Pipeline。


In [ ]:
# 先建立一個模擬的「巨型產線日誌 CSV」
mock_big_csv = "mock_production_stream.csv"
pd.DataFrame(
    {
        "event_id": range(1, 10001),
        "tool_status": np.random.choice(["RUN", "IDLE", "DOWN"], size=10000),
    }
).to_csv(mock_big_csv, index=False)

# --- 實作 Chunk 分批處理 Pipeline ---
print(">>> 啟動 Pandas Chunk 分批資料清洗管道...")

# 設定每個區塊只讀取 2000 行
chunk_size_setting = 2000
csv_reader_iterator = pd.read_csv(mock_big_csv, chunksize=chunk_size_setting)

# 迭代消耗這個 Reader 迭代器
for i, chunk_df in enumerate(csv_reader_iterator):
    # chunk_df 當前只包含了 2000 行資料，記憶體佔用極低
    print(f"正在處理第 {i + 1} 個區塊 (Chunk)，本批資料形狀: {chunk_df.shape}")

    # 在這裡執行高階清洗，例如：計算該區塊內 Tool 處於 DOWN 狀態的比例
    down_count = (chunk_df["tool_status"] == "DOWN").sum()
    print(f"檢測到機台當機事件數: {down_count} 筆")

# --- 清理實驗檔案 ---
if os.path.exists(mock_big_csv):
    os.remove(mock_big_csv)
print("\n 大數據分批讀取 Pipeline 執行完畢，全程維持超低記憶體安全水位。")

- 總結：在面對半導體產線秒級感測器噴出的巨量時間序列日誌（Equipment Logs）時，如果不做系統優化，系統很容易因為 OOM 崩潰。我的解決方針分為兩個層面：第一，儲存與讀取優化：我捨棄了傳統的 CSV 格式，改採 Apache Parquet 欄位式儲存。它內建 Snappy 高壓縮比，能節省高達 70% 以上的磁碟空間，且支援欄位裁剪（Column Projection），讓我在投遞特徵給機器學習模型時，能只讀取目標欄位，大幅減少 I/O 損耗。第二，資料管道流動優化：對於無法避免的巨型文字日誌，我會在 Python 中利用 Pandas 的 chunksize 參數將其包裝為迭代器，進行流式（Streaming）分批讀取與特徵重採樣（Resampling）聚合。這能確保大數據分析 Pipeline 不論在小型邊緣設備或雲端伺服器上運行，都能維持極低的記憶體安全水位。
